<a href="https://colab.research.google.com/github/Miranita-ar/Stroke-Analysis-Factor-/blob/main/Stroke_Analysis_Factoripynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IMPORT DATA

In [ ]:
install.packages("readxl")
library(readxl)




In [ ]:
data <- read_excel("/content/Data Stroke.xlsx")
data

# Pre Processing

In [ ]:
# Data Cleaning & EDA
summary(data)  # Ringkasan data
str(data) # cek tipe data


In [ ]:
# ubah data ke numeric
data$bmi <-as.numeric(data$bmi)
data$avg_glucose_level <-as.numeric(data$avg_glucose_level)
data$age <-as.numeric(data$age)

# Konversi variabel kategorik menjadi factor
data$gender <- as.factor(data$gender)
data$ever_married <- as.factor(data$ever_married)
data$work_type <- as.factor(data$work_type)
data$Residence_type <- as.factor(data$Residence_type)
data$smoking_status <- as.factor(data$smoking_status)
data$hypertension <- as.factor(data$hypertension)
data$heart_disease <- as.factor(data$heart_disease)

# Konversi faktor ke numeric biner
data$stroke <- as.numeric(as.character(data$stroke))
# Pastikan hanya ada nilai 0 dan 1
table(data$stroke)



In [ ]:
summary(data)

In [ ]:
# Menghapus baris dengan nilai "other" dalam kolom gender (Noise)
data <- data[data$gender != "Other", ]
# Cek apakah masih ada baris dengan "other"
table(data$gender)


#Cek missing value
colSums(is.na(data))

# Imputasi mv dengan mean
mean_bmi <- mean(data$bmi, na.rm = TRUE)
mean_bmi
data$bmi[is.na(data$bmi)] <- mean_bmi

summary(data) # cek lagi
str(data)


# ASUMSI MODEL AWAL

In [ ]:
install.packages("car")

In [ ]:
# variabel : gender, age, hypertension, heart_disease, ever_married, work_type, Residence_type, avg_glucose_level, bmi, smoking_status (ada 10)

# Asumsi Model Awal dengan interaksi
glm.logit <- glm(stroke ~ avg_glucose_level + Residence_type + work_type + ever_married + bmi + heart_disease*hypertension +age*heart_disease + age*hypertension + smoking_status*gender,
                 family = binomial(link = "logit"),data = data)
summary(glm.logit)

# Cek Multikolinearitas
library(car)
vif(glm.logit)
#alias(glm.logit)

# Model Selection
#Backward
step(glm.logit, direction = "backward")

# Stepwise
step(glm.logit, direction = "both")

# Forward
# Membuat model regresi logistik awal (tanpa prediktor)
glm.null <- glm(stroke ~ 1, family = binomial(link = "logit"), data = data)

# Melakukan seleksi maju menggunakan step()
step(glm.null, direction = "forward", scope = ~ gender + avg_glucose_level + Residence_type + ever_married + bmi +smoking_status + work_type + heart_disease*hypertension +age*heart_disease + age*hypertension)



In [ ]:
# Pakai model hasil forward karena AIC = 1600 (ASUMSI MODEL AWAL LANGSUNG DENGAN INTERAKSI)
glm.fit <- glm(stroke ~ avg_glucose_level + hypertension + age*heart_disease, family = binomial(link = "logit"), data = data)
summary(glm.fit)
vif(glm.fit)


glm.fit2 <- glm(stroke ~ avg_glucose_level + hypertension + age + heart_disease, family = binomial(link = "logit"), data = data)
summary(glm.fit2)
#vif(glm.fit2)

#LRT perbandingan model
anova(glm.fit2,glm.fit,  test = "Chisq")

#glm.fit3 <- glm(stroke ~ avg_glucose_level + hypertension + age , family = binomial(link = "logit"), data = data)
#summary(glm.fit3)
#vif(glm.fit3)

#LRT perbandingan model
#anova(glm.fit3,glm.fit,  test = "Chisq")


#LRT perbandingan model
#anova(glm.fit3,glm.fit2,  test = "Chisq")

#lrt
anova(glm.fit,glm.logit, test = "Chisq")

In [ ]:
glm.fit2 <- glm(stroke ~ avg_glucose_level + hypertension + age + heart_disease, family = binomial(link = "logit"), data = data)
summary(glm.fit2)
#vif(glm.fit2)

#LRT perbandingan model
anova(glm.fit2,glm.fit,  test = "Chisq")

# MODEL CHECKING

## Uji hosmer-lemeshow

In [ ]:
# uji hosmer-lemeshow (model fit jika p-value besar)
install.packages("ResourceSelection")


In [ ]:
library(ResourceSelection)
predicted_probabilities <- predict(glm.fit, type = "response")
hoslem.test(data$stroke, predicted_probabilities, g = 10)


## classification table ( sehat = 4861 , stroke = 249)

In [ ]:
install.packages("gmodels")

In [ ]:
# 1. Hasilkan prediksi probabilitas dari model
pred_prob <- predict(glm.fit, newdata = data, type = "response")

# 2. Konversi probabilitas ke kategori berdasarkan cutoff
cutoff <- 0.01  # Atur cutoff (misalnya, 0.5)
pred_class <- ifelse(pred_prob >= cutoff, 1, 0)

# 3. Pastikan pred_class adalah faktor dengan level yang sesuai
pred_class <- factor(pred_class, levels = c(0, 1))

# 4. Pastikan target aktual (data$stroke) juga faktor dengan level yang sama
data$stroke <- factor(data$stroke, levels = c(0, 1))

# 5. Buat confusion matrix
confusion_matrix <- table(Predicted = pred_class, Actual = data$stroke)
print(confusion_matrix)

# 6. Hitung metrik evaluasi jika confusion matrix valid
if (all(dim(confusion_matrix) == c(2, 2))) {
  TP <- confusion_matrix[2, 2] # True Positives
  TN <- confusion_matrix[1, 1] # True Negatives
  FP <- confusion_matrix[2, 1] # False Positives
  FN <- confusion_matrix[1, 2] # False Negatives

  accuracy <- (TP + TN) / sum(confusion_matrix)
  error <- (FP + FN) / sum(confusion_matrix)
  sensitivity <- ifelse((TP + FN) > 0, TP / (TP + FN), NA) # True Positive Rate
  specificity <- ifelse((TN + FP) > 0, TN / (TN + FP), NA) # True Negative Rate

  cat("Accuracy:", accuracy, "\n")
  cat("Error:", error, "\n")
  cat("Sensitivity:", sensitivity, "\n")
  cat("Specificity:", specificity, "\n")
} else {
  cat("Confusion matrix tidak lengkap. Periksa distribusi data prediksi atau data aktual.\n")
}


## Kurva ROC

In [ ]:
#kurva ROC
install.packages("pROC")
library(pROC)

In [ ]:
predicted_probabilities_roc <- predict(glm.fit, type = "response")
roc_curve <- roc(data$stroke,predicted_probabilities_roc)

# Plot kurva ROC
plot(roc_curve, col = "blue", lwd = 2, main = "ROC Curve")
abline(a = 0, b = 1, lty = 2, col = "red") # Garis diagonal (baseline)

# AUC
auc_value <- auc(roc_curve)
print(paste("AUC:", auc_value))

In [ ]:
install.packages("dplyr")

In [ ]:

install.packages("dplyr")

library(pROC)
library(dplyr)

# Hitung ROC Curve
roc_obj <- roc(data$stroke, data$pred_prob)

# Plot ROC Curve
plot(roc_obj, col = "blue", main = "ROC Curve", print.thres = TRUE)
abline(a = 0, b = 1, lty = 2, col = "red")

# Tentukan nilai cut-off optimal berdasarkan sensitivitas dan spesifisitas
coords_optimal <- coords(roc_obj, "best", ret = c("threshold", "sensitivity", "specificity"))
optimal_cutoff <- coords_optimal["threshold"]

# Tambahkan evaluasi dengan F1-score untuk setiap threshold
thresholds <- roc_obj$thresholds
f1_scores <- sapply(thresholds, function(threshold) {
  predicted <- ifelse(data$pred_prob >= threshold, 1, 0)
  tp <- sum(predicted == 1 & data$stroke == 1)
  fp <- sum(predicted == 1 & data$stroke == 0)
  fn <- sum(predicted == 0 & data$stroke == 1)

  precision <- ifelse(tp + fp > 0, tp / (tp + fp), 0)
  recall <- ifelse(tp + fn > 0, tp / (tp + fn), 0)
  f1 <- ifelse(precision + recall > 0, 2 * (precision * recall) / (precision + recall), 0)
  return(f1)
})

# Cari threshold dengan F1-score terbaik
best_f1_idx <- which.max(f1_scores)
best_f1_cutoff <- thresholds[best_f1_idx]
best_f1 <- f1_scores[best_f1_idx]

# Output nilai cut-off optimal dan metriknya
cat("Optimal Cut-off (based on Sensitivity/Specificity):", optimal_cutoff, "\n")
cat("Optimal Cut-off (based on F1-Score):", best_f1_cutoff, "\n")
cat("Best F1-Score:", best_f1, "\n")

# Plot F1-Score vs Threshold
plot(thresholds, f1_scores, type = "l", col = "blue", xlab = "Threshold", ylab = "F1-Score",
     main = "F1-Score vs Threshold")
abline(v = best_f1_cutoff, col = "

In [ ]:
# Tambahkan ke kurva ROC
#pred_obj <- prediction(pred, labels)
perf <- performance(pred_prob, "tpr", "fpr")

# Plot kurva ROC
plot(perf, colorize = TRUE, main = "ROC Curve with Cutoff")
abline(a = 0, b = 1, lty = 2, col = "gray") # Garis diagonal

# Tambahkan titik cutoff pada kurva
points(FPR, TPR, col = "red", pch = 19)
text(FPR, TPR, labels = paste("Cutoff =", cutoff), pos = 4)


In [ ]:
# Library yang diperlukan
install.packages('DescTools')

library(dplyr)
library(reshape2)
library(ggplot2)
library(DescTools)

# Variabel kategorik dan kontinu
categorical_vars <- c("gender", "hypertension", "heart_disease", "ever_married", "work_type", "Residence_type", "smoking_status")
continuous_vars <- c("age", "avg_glucose_level", "bmi")

# Fungsi untuk menghitung eta-squared (ukuran efek ANOVA)
eta_squared <- function(continuous_var, categorical_var) {
  model <- aov(continuous_var ~ categorical_var)
  eta2 <- summary(model)[[1]]$Sum Sq[1] / sum(summary(model)[[1]]$Sum Sq)
  return(eta2)
}

# Membuat dataframe untuk menyimpan hasil eta-squared
eta_squared_matrix <- data.frame(matrix(ncol = length(categorical_vars), nrow = length(continuous_vars)))
colnames(eta_squared_matrix) <- categorical_vars
rownames(eta_squared_matrix) <- continuous_vars

# Menghitung eta-squared untuk setiap kombinasi variabel kontinu dan kategorik
for (cont_var in continuous_vars) {
  for (cat_var in categorical_vars) {
    eta_squared_matrix[cont_var, cat_var] <- eta_squared(data[[cont_var]], data[[cat_var]])
  }
}

# Melihat matriks eta-squared
print(eta_squared_matrix)

# Visualisasi heatmap dari matriks eta-squared
heatmap_data <- melt(eta_squared_matrix)
colnames(heatmap_data) <- c("Continuous_Variable", "Categorical_Variable", "Eta_Squared")

ggplot(heatmap_data, aes(x = Continuous_Variable, y = Categorical_Variable, fill = Eta_Squared)) +
  geom_tile(color = "white") +
  scale_fill_gradient(low = "blue", high = "red", name = "Eta-Squared") +
  labs(title = "Matriks Korelasi Eta-Squared: Variabel Kontinu dan Kategorik",
       x = "Variabel Kontinu",
       y = "Variabel Kategorik") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))


# Data korelasi dalam format DataFrame
data = {
    'age': [0.0007701622, 0.07637885, 0.069578169, 0.46115563, 0.46526898, 1.968601e-04, 0.15994551],
    'avg_glucose_level': [0.0029945069, 0.03046421, 0.026213983, 0.02412713, 0.01284637, 2.287822e-05, 0.01073159],
    'bmi': [0.0006555921, 0.02564820, 0.001510526, 0.11260335, 0.19793147, 8.270679e-08, 0.07137097]
}

# Membuat DataFrame
correlation_df = pd.DataFrame(data, index=["gender", "hypertension", "heart_disease", "ever_married",
                                            "work_type", "Residence_type", "smoking_status"])

# Membuat heatmap menggunakan seaborn
plt.figure(figsize=(10, 6))
sns.heatmap(correlation_df, annot=True, cmap='coolwarm', center=0, linewidths=0.5)

# Menambahkan judul dan label sumbu
plt.title('Heatmap Korelasi antara Variabel Kontinu dan Kategorik')
plt.xlabel('Variabel Kontinu')
plt.ylabel('Variabel Kategorik')

# Menampilkan plot
plt.show()
hahahhaa